In [ ]:
# Practice Problems Day 18

### Author: Joanna Bieri (solutions)

## Introduction to Machine Learning

#### University of Redlands - DATA 301
#### Prof: Joanna Bieri [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
#### Day 18 Practice - SOLUTIONS
#### [Class Website](https://joannabieri.com/machine_learning.html)

---

**These are practice problems. They are not graded and you do not turn them in.** They are here so you can practice the new code from today: making the targets by moving the text over by one, making the causal mask, and building a tiny GPT that writes. Plan on about 45 minutes. Training a model takes a minute or two on a laptop, so be patient when you get to Problem 2!

**These are my solutions.** Try every problem yourself first, then come here to check your work. There is more than one right way to write the code, and your words will be different from mine, that is fine! Your generated text will be different from mine if anything about your run is different, that is fine too.

**Reading:** Geron chapter 15, **Building an English-to-Spanish Transformer**.

GOALS:

1. Build the input and the targets for next character prediction, and the causal mask.
2. Write the `forward` of a tiny GPT.
3. See what a shorter context window does to the model.
4. Explain in words how a GPT writes text.

**How much of this should you write yourself?** The setup cell is given: the text, `encode` and `decode`, the transformer block with the mask, the training function and `generate` from the notes. Problem 1 gives you most of the code, you write the missing lines. Problem 2 lists the steps. Problem 3 is words only. The **Optional exploration** at the end is extra, only if you are curious.

**Save your work.** Commit and push when you are done, so you have your code for later:

```bash
git add .
git commit -m "Day 18 practice"
git push
```

---

In [ ]:
# RUN THIS. Everything from the notes that you do not need to write today.
import math
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

text = open("data/shakespeare.txt").read()
chars = sorted(set(text))
vocab_size = len(chars)
char_to_id = {}
for i in range(vocab_size):
    char_to_id[chars[i]] = i

def encode(some_text):
    ids = []
    for c in some_text:
        ids.append(char_to_id[c])
    return ids

def decode(ids):
    pieces = []
    for i in ids:
        pieces.append(chars[i])
    return "".join(pieces)

def make_pieces(context_length):
    # cut the text into pieces of context_length + 1, then the last 10% is validation
    ids = torch.tensor(encode(text))
    n_pieces = len(ids) // (context_length + 1)
    pieces = ids[:n_pieces * (context_length + 1)].reshape(n_pieces, context_length + 1)
    X = pieces[:, :-1]
    y = pieces[:, 1:]
    n_train = int(0.9 * n_pieces)
    return X[:n_train], y[:n_train], X[n_train:], y[n_train:]

class TransformerBlock(nn.Module):
    def __init__(self, n_numbers=16, n_heads=2):
        super().__init__()
        self.attention = nn.MultiheadAttention(n_numbers, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(n_numbers)
        self.feed_forward = nn.Sequential(
            nn.Linear(n_numbers, 2 * n_numbers),
            nn.ReLU(),
            nn.Linear(2 * n_numbers, n_numbers))
        self.norm2 = nn.LayerNorm(n_numbers)

    def forward(self, X, mask=None):
        attention_out, weights = self.attention(X, X, X, attn_mask=mask)
        Z = self.norm1(X + attention_out)
        out = self.norm2(Z + self.feed_forward(Z))
        return out

def train_gpt(model, X_train, y_train, X_valid, y_valid, n_epochs):
    torch.manual_seed(42)
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
    train_losses = []
    valid_losses = []
    for epoch in range(n_epochs):
        model.train()
        total = 0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            scores = model(X_batch)
            loss = loss_fn(scores.reshape(-1, vocab_size), y_batch.reshape(-1))
            loss.backward()
            optimizer.step()
            total = total + loss.item()
        train_losses.append(total / len(train_loader))
        model.eval()
        with torch.no_grad():
            scores = model(X_valid)
            valid_loss = loss_fn(scores.reshape(-1, vocab_size), y_valid.reshape(-1)).item()
        valid_losses.append(valid_loss)
        print("epoch", epoch + 1, "  train loss:", round(train_losses[-1], 3), "  valid loss:", round(valid_loss, 3))
    return train_losses, valid_losses

def generate(model, start, n_new, temperature=1.0):
    ids = encode(start)
    model.eval()
    with torch.no_grad():
        for step in range(n_new):
            context = torch.tensor([ids[-model.context_length:]])   # the model remembers its own context length
            scores = model(context)
            probs = torch.softmax(scores[0, -1] / temperature, dim=0)
            ids.append(torch.multinomial(probs, num_samples=1).item())
    return decode(ids)

print("number of different characters:", vocab_size)

## Problem 1: The targets, the mask, and the forward pass

Let's start by building the three new pieces from today. These are the same steps as in the notes, so if you get stuck you can go back and look. But try it on your own first!

**What to copy, what to write:** Most of the code is given. Write the lines that the comments ask for.

**1a.** Remember that the text is its own label: the target is the input moved over by one character. Turn "to be or not" into ids with `encode`. Make the input (every id but the last) and the target (every id but the first). Print both with `decode` so you can check that the target really is the input moved over by one.

**1b.** Make the causal mask for 4 spots and print it. How many spots are True? What does True mean in this mask? (BEWARE: this is the one that is easy to get backwards!)

**1c.** Fill in the `forward` of the tiny GPT below. You need two things: make the mask, and then send `Z` through every block with a `for` loop. Then send 2 pieces of 64 characters through a brand new model and check the shape that comes out. Before you run it, make a guess! What should the three numbers in the shape be?

In [ ]:
# 1a
ids = encode("to be or not")
x_ids = ids[:-1]          # every id but the last
y_ids = ids[1:]           # every id but the first
print("input: ", repr(decode(x_ids)))
print("target:", repr(decode(y_ids)))

In [ ]:
# 1b
mask = torch.triu(torch.ones(4, 4, dtype=torch.bool), diagonal=1)
print(mask)
print("number of True spots:", mask.sum().item())

In [ ]:
# 1c
class TinyGPT(nn.Module):

    def __init__(self, context_length, n_numbers=64, n_heads=4, n_blocks=2):
        super().__init__()
        self.context_length = context_length                           # generate() reads this
        self.embed = nn.Embedding(vocab_size, n_numbers)
        self.position_embed = nn.Embedding(context_length, n_numbers)
        self.blocks = nn.ModuleList()
        for i in range(n_blocks):
            self.blocks.append(TransformerBlock(n_numbers, n_heads))
        self.out = nn.Linear(n_numbers, vocab_size)

    def forward(self, X):
        n_spots = X.shape[1]
        spots = torch.arange(n_spots)
        mask = torch.triu(torch.ones(n_spots, n_spots, dtype=torch.bool), diagonal=1)
        Z = self.embed(X) + self.position_embed(spots)
        for block in self.blocks:
            Z = block(Z, mask)
        scores = self.out(Z)
        return scores

X_train, y_train, X_valid, y_valid = make_pieces(context_length=64)
torch.manual_seed(42)
model = TinyGPT(context_length=64)
print("coming out:", model(X_train[:2]).shape)

**1a.** The input is `'to be or no'` and the target is `'o be or not'`. The target is the input moved over by one: after "t" comes "o", after "to" comes a space, and so on. Our piece has 12 characters, so we get 11 questions (the last character, "t", is only ever an answer, there is nothing after it to guess).

**1b.** 6 spots are True, the spots above the diagonal: $(4 \times 4 - 4) / 2 = 6$. True means **hidden**: the character in that row is NOT allowed to look at the character in that column, because it comes later. BEWARE: it is easy to think True means "allowed", but it is the other way around.

**1c.** The two missing pieces are `mask = torch.triu(torch.ones(n_spots, n_spots, dtype=torch.bool), diagonal=1)` and the loop `for block in self.blocks: Z = block(Z, mask)`. 2 pieces of 64 characters go in and `torch.Size([2, 64, 65])` comes out. Read it from left to right: 2 pieces, 64 spots in each piece, and 65 scores at every spot, one for every character that could come next.

---

## Problem 2: A shorter context window (steps outlined)

In the notes our tiny GPT could see 64 characters at once, and after 5 epochs the validation loss was **1.827**. The context window is how far back the model can look. So what happens if we make it smaller, and the model can only see 16 characters back? Let's try it and see!

**What to copy, what to write:** `make_pieces`, `train_gpt` and `generate` are in the setup cell, and `TinyGPT` is your class from Problem 1. Write the steps below yourself.

**2a.** Make the pieces with a context window of 16 characters and train a `TinyGPT` with `context_length=16` for 5 epochs. Is the validation loss better or worse than 1.827? Make a guess before you run it!

**2b.** Have your model write 300 characters starting with "JULIET:". Compare it to the text in the notes. Why would a model that can only see 16 characters back have a harder time? Think about what it can't see when it is in the middle of a line, eg. who is talking.

In [ ]:
# Steps 1 to 3
X_train_16, y_train_16, X_valid_16, y_valid_16 = make_pieces(context_length=16)
print("training pieces:", X_train_16.shape)

torch.manual_seed(42)
model_16 = TinyGPT(context_length=16)
losses_16 = train_gpt(model_16, X_train_16, y_train_16, X_valid_16, y_valid_16, n_epochs=5)

In [ ]:
# Step 4
torch.manual_seed(0)
print(generate(model_16, "JULIET:", n_new=300))

**2a.** After 5 epochs the validation loss is **1.872**, a little worse than the 1.827 in the notes. That is interesting, because the short model actually got MORE practice! Pieces of 17 characters means 59,049 training pieces instead of 15,443, so almost 4 times as many batches (and steps of Adam) every epoch. More practice, and it still did worse, because it can't see as far back.

**2b.** The text still looks like a play, with names in capitals, a colon and a new line, but there are more made up words. Your words will be different, here is one way to say it. A model that sees 16 characters back is usually in the middle of a line and can not see who is speaking, or even the start of the sentence it is in. With 64 characters it can often see the speaker's name and the whole line so far. Real chatbots have context windows of thousands of tokens or more, so they can keep track of a whole conversation.

---

## Problem 3: In words

Being able to explain this in plain words is a really useful skill. People are going to ask you how these tools work!

**3a.** A friend asks "how does ChatGPT write a whole answer?" Explain it to them using the words **next**, **probabilities** and **one at a time**. Use what we built today, not what you have heard about it.

**3b.** Chatbots sometimes write things that sound right but are false. Use what you learned today about how a GPT is trained and how it picks the next piece of text to explain why that can happen.

Your words will be different, here is one way to say each one.

**3a.** It never writes a whole answer at once. It looks at the text so far (your question plus whatever it has written already) and gives every possible next piece of text a probability. It picks one at random using those probabilities, adds it to the end, and does it again, one at a time, until it decides to stop. It is the same loop as our `generate` function, with tokens instead of characters and a MUCH bigger model.

**3b.** A GPT is trained to guess the next piece of text so that it looks like the text it was trained on. Nothing in that training checks whether a sentence is true. It learns what sounds right, and most of the time sounding right and being right go together, but not always. Then, when it writes, it picks each next piece at random from its probabilities, so it can go down a path that sounds great and is wrong, and it does not go back and check.

---

## Optional exploration

Only if you are curious. Nothing here is needed later.

**A.** Let's peek inside the model from Problem 1. First train it for 5 epochs, like in the notes. Then send in "ROMEO:\nWhat is th" and look at the probabilities for the next character. `torch.topk(probs, 5)` gives back the 5 biggest probabilities and their ids. Which 5 characters does the model like best? Do they make sense?

In [ ]:
# A: train the Problem 1 model like the notes (5 epochs), then look at its top 5 guesses
train_losses, valid_losses = train_gpt(model, X_train, y_train, X_valid, y_valid, n_epochs=5)

model.eval()
with torch.no_grad():
    scores = model(torch.tensor([encode("ROMEO:\nWhat is th")]))
probs = torch.softmax(scores[0, -1], dim=0)
top = torch.topk(probs, 5)
for k in range(5):
    print(repr(chars[top.indices[k].item()]), round(top.values[k].item(), 3))

**A.** After training, the top 5 guesses after "What is th" are `'e'` (0.45), `'a'` (0.267), `'o'` (0.14), `'i'` (0.094) and `'y'` (0.031). Those make "the", "that", "thou", "this" and "thy", all very common words in Shakespeare! The model learned this only from guessing the next character.